# FontSense — Google Colab Demo

**Student:** Rustamov Alixan

FontSense is a computer-vision capstone project that classifies a
cropped Latin-script text image into one of five broad categories:

- display
- handwriting
- monospace
- sans serif
- serif

> FontSense predicts broad font categories. It does **not** identify
> the exact font name or font family.

The frozen final CNN achieved a **test macro F1 of 0.8653** and a
**test accuracy of 86.67%** on the held-out test families. Those are
recorded final results; this demo does not rerun the test evaluation.

[Open this notebook in Google Colab](https://colab.research.google.com/github/rustamovalixan04-cyber/fontsense/blob/main/notebooks/07_colab_demo.ipynb)

## 1. Environment information

This cell reports the runtime. FontSense uses a small model, so CPU
inference is sufficient and a GPU is not required.

In [ ]:
import importlib.util
import platform
import subprocess
import sys

RUNNING_IN_COLAB = importlib.util.find_spec("google.colab") is not None

torch_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch; print(torch.cuda.is_available())",
    ],
    capture_output=True,
    text=True,
)
cuda_status = (
    torch_probe.stdout.strip()
    if torch_probe.returncode == 0
    else "PyTorch is not installed yet"
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Google Colab:", RUNNING_IN_COLAB)
print("CUDA available:", cuda_status)
print("CPU inference is sufficient for this FontSense demo.")

## 2. Obtain the repository

The notebook uses the real project repository:
`https://github.com/rustamovalixan04-cyber/fontsense.git`.
It clones the repository only when an existing FontSense checkout
cannot be found.

**Private-repository note:** if the repository is private, the person
running this notebook must already have GitHub access, or the
repository must be made public before the notebook is shared. This
notebook deliberately contains no password, personal access token, or
authentication cell.

In [ ]:
from pathlib import Path
import os

REPOSITORY_URL = (
    "https://github.com/rustamovalixan04-cyber/fontsense.git"
)


def find_existing_repository():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/fontsense"),
    ]
    for candidate in candidates:
        if (
            (candidate / ".git").is_dir()
            and (candidate / "app.py").is_file()
        ):
            return candidate.resolve()
    return None


REPOSITORY_ROOT = find_existing_repository()
if REPOSITORY_ROOT is None:
    clone_target = (
        Path("/content/fontsense")
        if RUNNING_IN_COLAB
        else Path.cwd() / "fontsense"
    )
    if clone_target.exists():
        raise RuntimeError(
            f"{clone_target} exists but is not a FontSense checkout. "
            "Move it or choose a clean runtime before continuing."
        )
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(clone_target)],
        check=True,
    )
    REPOSITORY_ROOT = clone_target.resolve()
else:
    print("Existing repository found; clone skipped.")

os.chdir(REPOSITORY_ROOT)
print("Repository ready:", REPOSITORY_ROOT)

## 3. Install the minimal demo dependencies

The project pins matching CPU-only builds of PyTorch and torchvision.
The remaining install contains only the libraries needed by the final
CNN interface. It does not install MLflow, notebook tooling, training
utilities, or the generated dataset.

In [ ]:
PYTORCH_CPU_INDEX = "https://download.pytorch.org/whl/cpu"
TORCH_PACKAGES = [
    "torch==2.13.0+cpu",
    "torchvision==0.28.0+cpu",
]
DEMO_PACKAGES = [
    "gradio>=4.44",
    "Pillow>=10.0",
    "numpy>=1.26,<3",
]

if RUNNING_IN_COLAB:
    install_commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--index-url",
            PYTORCH_CPU_INDEX,
            *TORCH_PACKAGES,
        ],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            *DEMO_PACKAGES,
        ],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-deps",
            "-e",
            ".",
        ],
    ]
    for command in install_commands:
        subprocess.run(command, check=True)
    print("Colab demo dependencies installed.")
else:
    print(
        "Local validation detected: using the existing environment. "
        "The install commands run automatically in Google Colab."
    )

In [ ]:
import gradio
import numpy
import PIL
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("gradio:", gradio.__version__)
print("Pillow:", PIL.__version__)
print("NumPy:", numpy.__version__)
print("CUDA used by this demo:", torch.version.cuda is not None)

## 4. Verify the frozen checkpoint

The demo must stop immediately if the tracked model is missing or if
even one byte differs from the frozen final checkpoint.

In [ ]:
import hashlib

EXPECTED_CHECKPOINT_SHA256 = (
    "c98cf0d1a02503a02b8f8242fec462ea"
    "2a0c455380238ec54fc4f62fdb13bb2f"
)
CHECKPOINT_PATH = (
    REPOSITORY_ROOT / "artifacts" / "cnn" / "cnn_model.pt"
)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Frozen CNN checkpoint is missing: {CHECKPOINT_PATH}"
    )

checkpoint_hash = hashlib.sha256(
    CHECKPOINT_PATH.read_bytes()
).hexdigest()
if checkpoint_hash != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Checkpoint hash mismatch. Stop: this is not the frozen "
        "final CNN."
    )

print("Checkpoint:", CHECKPOINT_PATH)
print("SHA-256:", checkpoint_hash)
print("Frozen checkpoint verification passed.")

## 5. Import the existing FontSense application

Prediction code is not copied into this notebook. The cell imports
`build_demo()` and the already-frozen predictor from `app.py`.

In [ ]:
import importlib

repository_text = str(REPOSITORY_ROOT)
if repository_text not in sys.path:
    sys.path.insert(0, repository_text)

fontsense_app = importlib.import_module("app")
demo = fontsense_app.build_demo()

print("Final model loaded:", fontsense_app.FINAL_CNN_PREDICTOR is not None)
print("Interface ready:", demo is fontsense_app.demo)

## 6. Technical smoke check

This cell creates one small image in memory and confirms the complete
inference path. Its prediction is only a software check and is **not**
evidence of model performance.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

EXPECTED_CLASSES = [
    "display",
    "handwriting",
    "monospace",
    "sans_serif",
    "serif",
]

smoke_image = Image.new("RGB", (224, 96), "white")
smoke_draw = ImageDraw.Draw(smoke_image)
smoke_draw.text(
    (18, 38),
    "FontSense technical check",
    fill="black",
    font=ImageFont.load_default(),
)

predictor = fontsense_app.FINAL_CNN_PREDICTOR
smoke_tensor = predictor.preprocess(smoke_image)
smoke_prediction = predictor.predict(smoke_image)
smoke_probabilities = smoke_prediction["probabilities"]

assert predictor.classes == EXPECTED_CLASSES
assert predictor.threshold == 0.60
assert tuple(smoke_tensor.shape) == (1, 48, 112)
assert list(smoke_probabilities) == EXPECTED_CLASSES
assert len(smoke_probabilities) == 5
assert abs(sum(smoke_probabilities.values()) - 1.0) < 1e-5

print("Model loaded: yes")
print("Class order:", predictor.classes)
print("Threshold:", predictor.threshold)
print("Preprocessing tensor shape:", tuple(smoke_tensor.shape))
print("Five probabilities returned: yes")
print("Probability sum:", sum(smoke_probabilities.values()))
print(
    "Smoke-test category:",
    smoke_prediction["predicted_category"],
    "(technical verification only)",
)

## 7. Use the demo

1. Run every notebook cell in order.
2. In the interface below, upload a cropped PNG or JPEG containing
   readable Latin text.
3. Press **Predict**.
4. Review the broad category, confidence percentage, all five
   probabilities, and accepted/uncertain status.
5. Use **Reset** before trying another image.

The 60% confidence threshold only decides whether the first guess is
accepted or marked uncertain. It does not change the model
probabilities.

In [ ]:
if RUNNING_IN_COLAB:
    demo.launch(
        share=True,
        inline=True,
        css=fontsense_app.APP_CSS,
    )
else:
    print(
        "Local validation mode: Colab launch skipped. "
        "Run `python app.py` for the local interface."
    )

### About the Colab link

Gradio normally displays the interface inline and may also create a
temporary public share link. The link is intended only for this
demonstration. It stops working when the Colab runtime is closed or
disconnected. Do not upload private or sensitive images to a shared
demo.

## 8. Limitations

- Exact font-family recognition is outside the current project scope.
- Screenshots and unfamiliar fonts can produce confident mistakes.
- Sans serif is the weakest category in the final evaluation.
- Synthetic rendered training images can differ from real designs,
  screenshots, scans, and photographs.
- An uncertain result is only a low-confidence first guess, not an
  exact identification.

This notebook performs inference only. It does not download the
3,600-image dataset, read train/validation/test manifests, train or
tune a model, or rerun the final test evaluation.